NEW VERSION

In [4]:
import pandas as pd
import requests
import io
import time
from tqdm import tqdm

# ==========================================
# LOAD NIFTY 500
# ==========================================

nse_url = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"

headers = {
    "User-Agent":
    "Mozilla/5.0"
}

nifty500 = pd.read_csv(nse_url)

tickers = nifty500["Symbol"].unique().tolist()

print("Companies:", len(tickers))

# ==========================================
# TARGET VARIABLES
# ==========================================

PNL_MAP = {
    "Sales": "Sales",
    "Operating Profit": "OperatingProfit",
    "Interest": "Interest",
    "Depreciation": "Depreciation",
    "Profit before tax": "PBT",
    "Net Profit": "NetProfit"
}

BS_MAP = {
    "Equity Capital": "EquityCapital",
    "Reserves": "Reserves",
    "Borrowings": "Borrowings",
    "Other Liabilities": "OtherLiabilities",
    "Total Liabilities": "TotalLiabilities",
    "Fixed Assets": "FixedAssets",
    "CWIP": "CWIP",
    "Investments": "Investments",
    "Other Assets": "OtherAssets",
    "Total Assets": "TotalAssets"
}

CF_MAP = {
    "Cash from Operating Activity": "CFO",
    "Free Cash Flow": "FCF"
}

# ==========================================
# HELPER
# ==========================================

def clean_label(x):

    return (
        str(x)
        .replace("\xa0", " ")
        .replace("+", "")
        .strip()
    )

# ==========================================
# SCRAPE
# ==========================================

panel_rows = []

failed = []

for ticker in tqdm(tickers):

    url = f"https://www.screener.in/company/{ticker}/consolidated/"

    try:

        r = requests.get(
            url,
            headers=headers,
            timeout=30
        )

        if r.status_code != 200:
            failed.append(ticker)
            continue

        tables = pd.read_html(
            io.StringIO(r.text)
        )

        for table in tables:

            try:

                first_col = table.columns[0]

                table[first_col] = (
                    table[first_col]
                    .astype(str)
                    .apply(clean_label)
                )

                table = table.set_index(first_col)

                cols = [str(c) for c in table.columns]

                # ======================================
                # ANNUAL P&L TABLE ONLY
                # ======================================

                required_rows = [
                    "Sales",
                    "Operating Profit",
                    "Profit before tax",
                    "Net Profit"
                ]

                if all(r in table.index for r in required_rows):

                    for metric, new_name in PNL_MAP.items():

                        if metric in table.index:

                            row = table.loc[metric]

                            for year in row.index:

                                panel_rows.append({
                                    "Ticker": ticker,
                                    "Year": str(year),
                                    "Metric": new_name,
                                    "Value": row[year]
                                })

                # ======================================
                # BALANCE SHEET
                # ======================================

                if "Total Assets" in table.index:

                    for metric, new_name in BS_MAP.items():

                        if metric in table.index:

                            row = table.loc[metric]

                            for year in row.index:

                                panel_rows.append({
                                    "Ticker": ticker,
                                    "Year": str(year),
                                    "Metric": new_name,
                                    "Value": row[year]
                                })

                # ======================================
                # CASH FLOW
                # ======================================

                if "Cash from Operating Activity" in table.index:

                    for metric, new_name in CF_MAP.items():

                        if metric in table.index:

                            row = table.loc[metric]

                            for year in row.index:

                                panel_rows.append({
                                    "Ticker": ticker,
                                    "Year": str(year),
                                    "Metric": new_name,
                                    "Value": row[year]
                                })

            except Exception:
                continue

        time.sleep(1)

    except Exception as e:

        failed.append(
            (ticker, str(e))
        )

# ==========================================
# LONG → WIDE
# ==========================================

panel = pd.DataFrame(panel_rows)

panel = (
    panel.pivot_table(
        index=["Ticker", "Year"],
        columns="Metric",
        values="Value",
        aggfunc="first"
    )
    .reset_index()
)

panel.columns.name = None

# ==========================================
# CLEAN YEAR
# ==========================================

panel["Year"] = (
    panel["Year"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
)

# ==========================================
# SAVE
# ==========================================

panel.to_csv(
    "screener_2.csv",
    index=False
)

print("\nSaved:")
print(panel.shape)

print("\nFailed:")
print(len(failed))

Companies: 500


100%|██████████| 500/500 [16:23<00:00,  1.97s/it]



Saved:
(8545, 20)

Failed:
0
